In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Import

In [ ]:
import sys
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
import os
import wandb
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
import joblib
from huggingface_hub import upload_file
from joblib import cpu_count
import numpy as np

In [ ]:
n_jobs = max(1, cpu_count() - 1)

# Consts

In [ ]:
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')
FINAL_MODEL_PATH = os.path.join(os.getcwd(), 'models', 'model.joblib')

ESTIMATORS = [
    LinearSVC(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
    RandomForestClassifier(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
    LogisticRegression(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
]

ESTIMATOR_NUMBER = 0 # 0 - LinearSVC, 1 - RandomForestClassifier, 2 - LogisticRegression
ESTIMATOR = ESTIMATORS[ESTIMATOR_NUMBER]

# Pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))),
    ("clf", ESTIMATOR),
])

# Datasets

In [ ]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
test_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_test.csv"))

In [ ]:
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"].map({'F': 0, 'M': 1})
x_test, y_test = test_df["text"], test_df["gender_label"].map({'F': 0, 'M': 1})

# Wandb init

In [ ]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)
run = wandb.init(project="who-wrote-it-nlp", name="training", entity="who-wrote-it-nlp", job_type="training",group="training")

# Hyperparameters

In [ ]:
# set best params from hyperparameter tuning
best_params = {}

In [ ]:
wandb.config.update(best_params)

In [ ]:
pipeline.set_params(**best_params)

# Training

In [ ]:
pipeline.fit(x_train, y_train)

# Evaluation on test set

In [ ]:
test_pred = pipeline.predict(x_test)
print(classification_report(y_test, test_pred))

In [ ]:
cm = confusion_matrix(y_test, test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['F', 'M'])
disp.plot(cmap='Blues',)
plt.title('Confusion Matrix on Test Set')
plt.show()

wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    y_true=y_test.tolist(), 
    preds=test_pred.tolist() #type: ignore
)})

run.finish()

# Overfitting check

In [ ]:
train_accuracy = pipeline.score(x_train, y_train)
test_accuracy = pipeline.score(x_test, y_test)

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Difference (overfitting indicator): {train_accuracy - test_accuracy:.4f}")

if train_accuracy - test_accuracy > 0.05:
    print("Model shows signs of overfitting")
elif test_accuracy - train_accuracy > 0.05:
    print("Model shows signs of underfitting")
else:
    print("Model is well-balanced")

# Save model

## Locally

In [ ]:
joblib.dump(pipeline,  FINAL_MODEL_PATH)

## In Hugging Face Hub

In [ ]:
hg_token = os.getenv("HUGGINGFACE_HUB_TOKEN")
if not hg_token:
    raise ValueError("Please set the HUGGINGFACE_HUB_TOKEN environment variable to upload the model to Hugging Face Hub.")
upload_file(
    path_or_fileobj=FINAL_MODEL_PATH,
    path_in_repo=f"model_{ESTIMATOR.__class__.__name__}.joblib",
    repo_id="qg2020252627/twitter_author_profiling_by_gender_nlp",
    repo_type="model",
    token=hg_token
)